In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib 

In [2]:
df = pd.read_csv("/Users/kaviraj/Desktop/GUVI/Project3/regression/r_feature_engineering.csv")

# convert date from int to datetime
df['appointment_date_continuous'] = pd.to_datetime(df['appointment_date_continuous'])

# int to bool
bool_cols = df.select_dtypes(
    include='bool'
).columns

df[bool_cols] = df[
    bool_cols
].astype(int)

# Train Data -> from 2020, Jan to 2021, Feb
# Test data -> 2021, March

train_df = df[
    df['appointment_date_continuous'] < '2021-03-01'
]

test_df = df[
    (df['appointment_date_continuous'] >= '2021-03-01') &
    (df['appointment_date_continuous'] < '2021-04-01')
]

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

forecast_features = [
    "lag_1",
    "lag_7",
    "lag_30",

    "rolling_mean_7",
    "rolling_mean_30",

    "specialty_psychotherapy",
    "specialty_speech therapy",
    "specialty_physiotherapy",
    "specialty_Unknown",
    "specialty_occupational therapy",

    "day_of_week_Monday",
    "day_of_week_Saturday",
    "day_of_week_Sunday",
    "day_of_week_Thursday",
    "day_of_week_Tuesday",
    "day_of_week_Wednesday"
]

# Create train data
X_train = train_df[forecast_features]

y_train = train_df['daily_appointments']

# Create test data

X_test = test_df[forecast_features]

y_test = test_df['daily_appointments']


# Model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

lr = LinearRegression()

lr.fit(
    X_train,
    y_train
)

y_pred_lr = lr.predict(X_test)

Train Shape: (425, 39)
Test Shape: (31, 39)


In [3]:
print("Linear Regression")
print("-"*30)

print(
    "MAE:",
    mean_absolute_error(
        y_test,
        y_pred_lr
    )
)

print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(
            y_test,
            y_pred_lr
        )
    )
)

print(
    "R2:",
    r2_score(
        y_test,
        y_pred_lr
    )
)

Linear Regression
------------------------------
MAE: 3.415600011207965
RMSE: 5.741166511237769
R2: 0.9995420557307715


In [25]:
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred_lr
})

results.to_csv("/Users/kaviraj/Desktop/GUVI/Project3/streamlit_app/actual_vs_predicted.csv", index=False)

joblib.dump(
    lr,
    "/Users/kaviraj/Desktop/GUVI/Project3/streamlit_app/lr_demand_forecast_model.pkl"
)

joblib.dump(
    X_train.columns.tolist(),
    "/Users/kaviraj/Desktop/GUVI/Project3/streamlit_app/lr_columns.pkl"
)

['/Users/kaviraj/Desktop/GUVI/Project3/streamlit_app/lr_columns.pkl']

In [5]:
comparison_df = pd.DataFrame({
    "Date": test_df["appointment_date_continuous"],
    "Actual": y_test,
    "Predicted": np.round(y_pred_lr, 2)
})

print(comparison_df.head(20))

          Date  Actual  Predicted
425 2021-03-01     112     111.47
426 2021-03-02      66      69.02
427 2021-03-03     211     209.54
428 2021-03-04      63      63.43
429 2021-03-05      92      93.96
430 2021-03-06      93      90.11
431 2021-03-07      51      52.33
432 2021-03-08      34      33.88
433 2021-03-09     549     549.86
434 2021-03-10     432     431.09
435 2021-03-11     419     416.47
436 2021-03-12     340     338.40
437 2021-03-13     769     751.75
438 2021-03-14      45      40.76
439 2021-03-15     102     103.69
440 2021-03-16     189     194.95
441 2021-03-17     139     137.65
442 2021-03-18     237     239.69
443 2021-03-19     125     125.20
444 2021-03-20     217     217.54
